In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampNTZType, TimestampType, DoubleType, LongType

import pandas as pd

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [13]:
df_green = spark.read.parquet("data/pq/green/*/*")
df_green.createOrReplaceTempView("green")
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [14]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-24 09:00:00|  81|             59.49|             2|
|2020-01-04 21:00:00|  25|            513.83|            32|
|2020-01-10 19:00:00|  66| 545.6800000000002|            27|
|2020-01-30 07:00:00|  75| 556.6600000000001|            40|
|2020-01-18 01:00:00| 260|            144.56|            12|
|2020-01-12 08:00:00| 177|31.090000000000003|             2|
|2020-01-20 21:00:00| 166|            133.28|            12|
|2020-01-03 04:00:00|  14|            105.34|             2|
|2020-01-30 20:00:00|  74| 766.4400000000002|            58|
|2020-01-02 16:00:00|  66|             257.0|            12|
|2020-01-11 23:00:00|  42|            340.47|            31|
|2020-01-13 12:00:00|  89|            135.25|             6|
|2020-01-02 08:00:00| 174|50.309999999999995|             3|
|2020-01-20 01:00:00|  4

In [35]:
df_green.write.parquet("data/report/revenue/green", mode="overwrite")

In [15]:
df_yellow = spark.read.parquet("data/pq/yellow/*/*")
df_yellow.createOrReplaceTempView("yellow")
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [16]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_yellow_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-03 19:00:00| 142| 6023.089999999995|           403|
|2020-01-26 14:00:00| 239| 6541.649999999994|           437|
|2020-01-09 01:00:00| 100|            653.56|            37|
|2020-01-31 18:00:00| 237|12689.400000000009|           810|
|2020-01-04 03:00:00| 246|2092.5400000000004|           121|
|2020-01-14 09:00:00| 239| 4882.359999999997|           298|
|2020-01-27 16:00:00| 162| 7989.979999999996|           452|
|2020-01-17 20:00:00| 170| 6884.189999999997|           407|
|2020-01-23 15:00:00| 142| 5378.829999999997|           341|
|2020-01-27 06:00:00|  24|            536.49|            23|
|2020-01-01 04:00:00| 170|            2306.2|           135|
|2020-01-05 12:00:00|  68|3495.9599999999987|           235|
|2020-01-13 17:00:00| 107|4066.6799999999976|           241|
|2020-01-21 19:00:00| 16

In [39]:
df_yellow_revenue.repartition(20).write.parquet("data/report/revenue/yellow", mode="overwrite")

In [22]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed("amount", "green_amount") \
    .withColumnRenamed("number_records", "green_number_records")

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed("amount", "yellow_amount") \
    .withColumnRenamed("number_records", "yellow_number_records")

In [23]:
# now lets join them together (by zone and hour)
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=["hour", "zone"], how="outer")

In [24]:
df_join.write.parquet("data/report/revenue/joined", mode="overwrite")

In [25]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  65|            199.49|                  10|            409.35|                   19|
|2020-01-01 00:00:00| 161|              NULL|                NULL| 9410.210000000001|                  488|
|2020-01-01 01:00:00|   3|46.230000000000004|                   2|              NULL|                 NULL|
|2020-01-01 02:00:00|  17|200.03000000000003|                  11|315.56999999999994|                   16|
|2020-01-01 02:00:00| 107|              NULL|                NULL|7876.8699999999935|                  485|
|2020-01-01 02:00:00| 162|              NULL|                NULL| 3101.869999999999|                  180|
|2020-01-01 03:00:00| 234|  

In [27]:
# now lets do one big table and one small table
!powershell -Command "Invoke-WebRequest -Uri 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv' -OutFile 'taxi_zone_lookup.csv'"

In [29]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [35]:
df_join = spark.read.parquet("data/report/revenue/joined/*")

In [ ]:
# lets join df join and zones
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID, how="left").drop("Zone") # drop zone name, its causing issue when saving due same name col
df_result.show()

+-------------------+------------------+--------------------+------------------+---------------------+----------+---------+------------+
|               hour|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|service_zone|
+-------------------+------------------+--------------------+------------------+---------------------+----------+---------+------------+
|2020-01-01 00:00:00|             70.12|                   4|              31.0|                    1|       228| Brooklyn|   Boro Zone|
|2020-01-01 00:00:00|              NULL|                NULL| 4599.189999999998|                  219|       246|Manhattan| Yellow Zone|
|2020-01-01 03:00:00|              NULL|                NULL|2977.1899999999996|                  154|       163|Manhattan| Yellow Zone|
|2020-01-01 04:00:00|              NULL|                NULL| 2907.449999999999|                  147|       114|Manhattan| Yellow Zone|
|2020-01-01 06:00:00|             18.85| 

In [41]:
df_result.write.parquet("data/report/revenue/joined_with_zones", mode="overwrite")